# J15a — Première application Streamlit de géomarketing en Île-de-France

## Objectifs

- Transformer les sorties contrôlées de J14 en une application lisible par un client.
- Explorer les communes et tous les IRIS IDF : population, restauration, revenus disponibles et transports.
- Cliquer sur toute commune du profil IDF, afficher ses IRIS et consulter un IRIS.
- Présenter les lignes métro, bus, RER et Transilien autour de la commune sélectionnée, avec couleurs et pictogrammes.
- Proposer des filtres et des téléchargements de tableaux et fiches descriptives.
- Garder les dates, les statuts de disponibilité et les unités des indicateurs.
- Préparer une première démonstration en ligne depuis GitHub, après vérification locale.

## Périmètre

Communes et IRIS de l’ensemble du profil IDF, selon la géographie J14.
La carte régionale permet de cliquer sur une commune pour afficher sa sous-couche
IRIS ; un clic sur un IRIS ouvre sa fiche. Les lignes, arrêts et restaurants
proviennent des exports régionaux J14/J13c, et non du seul focus Cergy–Pontoise.
La géographie du profil peut différer de la géographie administrative actuelle.
L’application reprend J13e si J14 a été exécuté à partir de J13e.

## Interprétation

Une démonstration peut commencer dès J15a : elle décrit les données disponibles.
Les calculs d’horaires et de fréquences proposés après J14 deviennent **J15b**.
Ils enrichiront la même application ; leur absence ne bloque pas cette première version.
La validation terrain et l’étude d’un emplacement précis restent nécessaires pour une recommandation commerciale.
Cette version ne calcule ni rentabilité, ni trafic de voyageurs, ni score de viabilité.

## Utilisation

Télécharger aussi `j15_streamlit_geomarketing_idf.zip` : ce notebook prépare les
données, les fichiers Python du ZIP construisent l’application. Copier les nouveaux
fichiers dans le projet en conservant les fichiers existants. Il n’y a aucune donnée
de test ou étude fictive dans le ZIP. Sans exports réels, l’application indique simplement
que les données restent à préparer.

Enregistrer le notebook dans `notebooks/15a_preparer_application_streamlit_idf.ipynb`.
Adapter le bloc 2, exécuter jusqu’au bloc 13, puis lancer Streamlit depuis Anaconda Prompt.
Les sources nationales restent dans raw ; l’application lit les petits exports J15a.
Le déploiement est expliqué, mais il n’est pas effectué par ce notebook.


## 1 — Imports et environnement

Réutiliser l’environnement J14, avec pandas, GeoPandas et Plotly, pour préparer
les contours régionaux. L’application lit ensuite des CSV/GeoJSON et dispose de
ses dépendances dans app/requirements.txt. GeoPandas est nécessaire au notebook.


In [1]:
import os
for variable in ['OMP_NUM_THREADS','NUMEXPR_NUM_THREADS','MKL_NUM_THREADS']:os.environ[variable]='1'
from pathlib import Path
from datetime import datetime,timezone
import json,csv,hashlib,shutil,sys
import numpy as np
import pandas as pd
import geopandas as gpd
from IPython.display import display
pd.set_option('display.max_columns',50)
print('pandas',pd.__version__)


pandas 2.2.2


## 2 — Chemins et fichiers à reprendre


Adapter la racine. Les fichiers sont des sorties **J14 contrôlées**, pas les fichiers
bruts nationaux. Le ZIP doit fournir app/core.py, app/streamlit_app.py et les ressources
graphiques. Le fichier de points J13c sert à reprendre les restaurants géolocalisés.
Si aucun fichier de points n’est disponible, CHARGER_RESTAURANTS=False conservera
les comptes existants et une carte sans marqueurs de restaurants.

Les nouvelles entrées sont les GPKG régionaux et les CSV régionaux du bloc 26 J14.
Extraire le ZIP à la racine du projet avant les contrôles. Le registre peut être
hors OneDrive avec GEOMARKETING_REGISTRE ou un chemin explicite. Son séparateur
est détecté par les six noms de colonnes, sans ignorer un fichier illisible.


In [2]:
RACINE=Path(os.environ.get('GEOMARKETING_RACINE',r'C:\Users\almou\OneDrive\GeoMarketing_IDF'))
P14=RACINE/'data/processed/j14'
A14=RACINE/'data/app/j14'
FICHIER_IRIS=P14/'profil_iris_idf_j14.csv'
FICHIER_COMMUNES=P14/'profil_communes_idf_j14.csv'
FICHIER_META=P14/'metadata_transports_j14.json'
FICHIER_MANIFEST14=A14/'manifest_transports_j14.json'
FICHIER_RELATIONS=P14/'relations_arrets_lignes_j14.csv'
FICHIER_ARRETS_REGION=P14/'arrets_idfm_idf_et_abords_j14.csv'
FICHIER_CATALOGUE_REGION=P14/'catalogue_lignes_idfm_j14.csv'
FICHIER_GEO_IRIS=P14/'profil_iris_idf_j14.gpkg'
FICHIER_GEO_COMMUNES=P14/'profil_communes_idf_j14.gpkg'
FICHIER_TRACES_REGION=P14/'traces_transports_j14.gpkg'
COUCHE_IRIS='profils_iris'
COUCHE_COMMUNES='profils_communes'
COUCHE_TRACES='traces'
SIMPLIFICATION_CARTE_M=60  # Affichage uniquement ; les calculs J14 restent ceux des contours détaillés.
FICHIER_POINTS=RACINE/'data/processed/j13c/points_affectations_idf_j13c.csv'
FICHIER_META_C=RACINE/'data/processed/j13c/metadata_affectations_iris_j13c.json'
FICHIER_REGISTRE=Path(os.environ.get('GEOMARKETING_REGISTRE',RACINE/'docs/sources.csv'))
SEPARATEUR_REGISTRE=None  # Auto-détection virgule / point-virgule ; ou renseigner ',' / ';'.
DOSSIER_APP=RACINE/'data/app/j15'
DOSSIER_INTERIM=RACINE/'data/interim/j15a'
DOSSIER_DOCS=RACINE/'docs'
CHARGER_RESTAURANTS=True
MAX_FICHIER_MO=20
COLONNES_SOURCE_REVENUS={'MEDIANE_UC':'J13B_NIVEAU_VIE_MEDIAN_ANNUEL_UC_EUR','PAUVRETE_PCT':'J13B_TAUX_PAUVRETE_60_PCT',
    'STATUT_REVENUS':'J13B_STATUT_REVENUS','ANNEE_REVENUS':'J13B_ANNEE_FILOSOFI_SOURCE','GEOGRAPHIE_REVENUS':'J13B_GEOGRAPHIE_FILOSOFI_SOURCE'}
for p in [DOSSIER_APP,DOSSIER_INTERIM,DOSSIER_DOCS]:p.mkdir(parents=True,exist_ok=True)
print('Projet',RACINE)


Projet C:\Users\almou\OneDrive\GeoMarketing_IDF


## 3 — Lectures et exports contrôlés


Les codes géographiques restent du texte. Les CSV amont produits par J14 sont
attendus avec une virgule ; le registre des sources utilise `;`. Les nombres
manquants restent NA. Une colonne facultative absente sera signalée dans le diagnostic.
La sortie ne peut pas écraser une entrée.


In [3]:
def lire_csv(p,sep=','):
    with Path(p).open(encoding='utf-8-sig',newline='') as f:
        reader=csv.reader(f,delimiter=sep,strict=True);cols=next(reader)
        if len(cols)<2 or len(cols)!=len(set(cols)):raise ValueError(str(p)+' : en-tête/séparateur invalide.')
        for ligne in reader:
            if ligne and len(ligne)!=len(cols):raise ValueError(str(p)+' : ligne '+str(reader.line_num)+' mal formée.')
    return pd.read_csv(p,sep=sep,dtype='string',encoding='utf-8-sig')
def nombre(s):
    t=s.astype('string').str.strip().replace('',pd.NA).str.replace('\u202f','',regex=False).str.replace('\u00a0','',regex=False)
    t=t.str.replace(' ','',regex=False).str.replace(',','.',regex=False)
    n=pd.to_numeric(t,errors='coerce').astype(float)
    if (t.notna()&(n.isna()|~np.isfinite(n))).any():raise ValueError('Nombre non interprétable : '+str(t[t.notna()&n.isna()].head().tolist()))
    return n
def sha(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()
SORTIES=[];DIAGNOSTIC=[]
def exporter(t,nom,sep=',',dossier=None):
    p=(dossier or DOSSIER_APP)/nom
    if p.resolve() in {f.resolve() for f in ENTREES}:raise ValueError('Sortie identique à une entrée.')
    tmp=p.with_suffix('.temp');t.to_csv(tmp,index=False,sep=sep,encoding='utf-8-sig');tmp.replace(p)
    if p not in SORTIES:SORTIES.append(p)
    return p
def extraire(table,source,numerique=True):
    if source not in table:
        DIAGNOSTIC.append({'COLONNE_SOURCE':source,'STATUT':'ABSENTE_INDICATEUR_INDISPONIBLE'})
        return pd.Series(np.nan if numerique else pd.NA,index=table.index)
    return nombre(table[source]) if numerique else table[source]

def lire_registre(p,sep=None):
    import io
    requis=['fichier','source','annee_donnees','annee_geographie','date_telechargement','url']
    try:
        contenu=Path(p).read_bytes().decode('utf-8-sig')
    except OSError as e:
        raise OSError(f"Registre illisible : {p}. Rendre le fichier disponible localement ou renseigner une copie hors OneDrive au bloc 2.") from e
    if not contenu.strip():raise ValueError('Le registre est vide : '+str(p))
    candidats=[]
    for delim in ([sep] if sep else [',',';','\t']):
        lecteur=csv.reader(io.StringIO(contenu),delimiter=delim,strict=True)
        cols=[x.strip() for x in next(lecteur)]
        if set(requis).issubset(cols):candidats.append((delim,cols))
    if len(candidats)!=1:raise ValueError('Séparateur/en-tête du registre non reconnu. Renseigner SEPARATEUR_REGISTRE au bloc 2.')
    delim,cols=candidats[0]
    if len(cols)!=len(set(cols)):raise ValueError('Colonnes du registre dupliquées.')
    for row in csv.reader(io.StringIO(contenu),delimiter=delim,strict=True):
        if row and len(row)!=len(cols):raise ValueError('Registre : nombre de champs incohérent.')
    table=pd.read_csv(io.StringIO(contenu),sep=delim,dtype='string')
    table.columns=[c.strip() for c in table.columns]
    print('Séparateur du registre :',repr(delim))
    return table[requis].copy()


## 4 — Contrôler les entrées et le manifeste J14


Les empreintes du manifeste permettent de vérifier que les contours, arrêts et
lignes appartiennent à la même génération des exports. Si les sorties J14 ont
changé, terminer de nouveau J14 avant de préparer J15a.


In [4]:
ENTREES=[FICHIER_IRIS,FICHIER_COMMUNES,FICHIER_META,FICHIER_MANIFEST14,FICHIER_RELATIONS,FICHIER_ARRETS_REGION,FICHIER_CATALOGUE_REGION,FICHIER_GEO_IRIS,FICHIER_GEO_COMMUNES,FICHIER_TRACES_REGION]
fichiers14=['iris_focus_j14.geojson','traces_focus_j14.geojson']
ENTREES += [A14/n for n in fichiers14]
if CHARGER_RESTAURANTS:ENTREES.append(FICHIER_POINTS)
if FICHIER_META_C.is_file():ENTREES.append(FICHIER_META_C)
if FICHIER_REGISTRE.is_file():ENTREES.append(FICHIER_REGISTRE)
for p in ENTREES:
    if not p.is_file():raise FileNotFoundError(str(p))
for n in ['app/core.py','app/streamlit_app.py','app/assets/world_110m.json','app/assets/pictogrammes_idfm.json','app/territory_map.py','app/territory_component/index.html','app/territory_component/plotly.min.js','app/territory_component/world_110m.json']:
    if not (RACINE/n).is_file():raise FileNotFoundError('Extraire les fichiers du ZIP dans le projet : '+n)
meta14=json.loads(FICHIER_META.read_text(encoding='utf-8'))
m14=json.loads(FICHIER_MANIFEST14.read_text(encoding='utf-8'))
if meta14.get('etape')!='J14_TRANSPORTS_MULTIMODES':raise ValueError('Métadonnées J14 non reconnues.')
if meta14['codes_focus']!=m14['codes_focus']:raise ValueError('Focus différent entre les exports J14.')
index14={r['FICHIER']:r['SHA256'] for r in m14['fichiers']}
for nom in fichiers14:
    if nom not in index14 or sha(A14/nom)!=index14[nom]:raise ValueError(nom+' : fichier différent du manifeste J14.')
src=lire_csv(FICHIER_IRIS);communes_source=lire_csv(FICHIER_COMMUNES)
if not src['IRIS'].str.fullmatch(r'\d{9}',na=False).all() or not src['IRIS'].is_unique:raise ValueError('Codes IRIS invalides/dupliqués.')
if set(src['IRIS'].str[:2])!={'75','77','78','91','92','93','94','95'}:raise ValueError('Conserver tous les départements IDF.')
if not communes_source['CODGEO'].str.fullmatch(r'\d{5}',na=False).all() or not communes_source['CODGEO'].is_unique:raise ValueError('Codes communaux invalides/dupliqués.')
ATTENDUS=set(src['IRIS']);NOMBRE_IRIS=len(src)
print('IRIS conservés',NOMBRE_IRIS,'; profils issus de',meta14['etape_base'])


IRIS conservés 5264 ; profils issus de J13D


## 5 — Préparer les indicateurs IRIS pour le public


Les noms des colonnes de l’application sont courts ; app/core.py contient leurs
libellés clients. Seules les variables utiles à cette première version sont
exportées. Les profils complets J14 restent dans processed. Les nombres nuls
présents dans la source restent nuls ; les données absentes ne deviennent pas zéro.


In [5]:
iris=pd.DataFrame({'IRIS':src['IRIS'],'CODGEO':src['J13D_COMMUNE_CONTEXTE']})
nom_commune=communes_source.set_index('CODGEO')['NOM_COMMUNE']
iris['NOM_COMMUNE']=iris['CODGEO'].map(nom_commune)
iris['NOM_IRIS']=extraire(src,'NOM_IRIS',False).fillna(src['IRIS'])
iris['TYP_IRIS']=extraire(src,'J13A_LIBELLE_TYPE_IRIS',False).fillna(extraire(src,'TYP_IRIS',False)).fillna('Type non renseigné')
mapping={'POPULATION':'J13A_POPULATION','PART_1529_PCT':'J13A_PART_1529_PCT',
    'RESTAURANTS':'J13C_NB_OBS_RESTAURATION','RAPIDES':'J13C_NB_OBS_RESTAURATION_RAPIDE'}
for col,source in mapping.items():iris[col]=extraire(src,source)
iris['DENSITE_RESTAURANTS']=iris['RESTAURANTS'].div(iris['POPULATION'].where(iris['POPULATION'].gt(0)))*10000
for col,source in COLONNES_SOURCE_REVENUS.items():iris[col]=extraire(src,source,col!='STATUT_REVENUS')
iris['STATUT_REVENUS']=iris['STATUT_REVENUS'].fillna('Source fiscale indisponible')
for mode,rayon in meta14['rayons_metres'].items():
    mapping={f'DIST_{mode}_M':f'J14_DIST_POINT_REF_ARRET_OBS_{mode}_M',
        f'LIGNES_{mode}':f'J14_NB_LIGNES_OBS_{mode}_RAYON_{rayon}M',
        f'SURFACE_{mode}_PCT':f'J14_PART_SURFACE_IRIS_PROCHE_ARRET_OBS_{mode}_PCT',
        f'ARRETS_{mode}':f'J14_NB_ENTITES_ARRET_OBS_AFFECTEES_{mode}',
        f'STATUT_{mode}':f'J14_STATUT_OBSERVATIONS_{mode}'}
    for col,source in mapping.items():iris[col]=extraire(src,source,not col.startswith('STATUT_'))
assert set(iris['IRIS'])==ATTENDUS and len(iris)==NOMBRE_IRIS
if iris['NOM_COMMUNE'].isna().any():raise ValueError('Communes IRIS non appariées.')
exporter(iris,'profil_iris.csv')
display(iris.head())


,IRIS,CODGEO,NOM_COMMUNE,NOM_IRIS,TYP_IRIS,POPULATION,PART_1529_PCT,RESTAURANTS,RAPIDES,DENSITE_RESTAURANTS,MEDIANE_UC,PAUVRETE_PCT,STATUT_REVENUS,ANNEE_REVENUS,GEOGRAPHIE_REVENUS,DIST_METRO_M,LIGNES_METRO,SURFACE_METRO_PCT,ARRETS_METRO,STATUT_METRO,DIST_BUS_M,LIGNES_BUS,SURFACE_BUS_PCT,ARRETS_BUS,STATUT_BUS,DIST_RER_M,LIGNES_RER,SURFACE_RER_PCT,ARRETS_RER,STATUT_RER,DIST_TRANSILIEN_M,LIGNES_TRANSILIEN,SURFACE_TRANSILIEN_PCT,ARRETS_TRANSILIEN,STATUT_TRANSILIEN
0,751031101,75056,Paris,Les Archives 1,Habitat,2317.001126,26.952834,16.0,7.0,69.054779,37610.0,12.0,DISPONIBLE,2021.0,2022.0,127.585700,5.0,100.0,1.0,OBSERVE_COUVERTURE_A_VERIFIER,98.285103,5.0,100.00000,2.0,OBSERVE_COUVERTURE_A_VERIFIER,1409.056951,0.0,0.00000,0.0,OBSERVE_COUVERTURE_A_VERIFIER,2002.783861,0.0,0.0,0.0,OBSERVE_COUVERTURE_A_VERIFIER
1,751051702,75056,Paris,Saint-Victor 2,Habitat,1665.564957,31.013082,22.0,13.0,132.087313,37090.0,11.0,DISPONIBLE,2021.0,2022.0,137.781303,2.0,100.0,0.0,OBSERVE_COUVERTURE_A_VERIFIER,151.438370,6.0,100.00000,1.0,OBSERVE_COUVERTURE_A_VERIFIER,1001.134657,0.0,0.00000,0.0,OBSERVE_COUVERTURE_A_VERIFIER,1693.086902,0.0,0.0,0.0,OBSERVE_COUVERTURE_A_VERIFIER
2,751062309,75056,Paris,Notre-Dame des Champs 9,Habitat,2414.995652,27.831465,25.0,10.0,103.519855,35410.0,13.0,DISPONIBLE,2021.0,2022.0,185.537041,4.0,100.0,4.0,OBSERVE_COUVERTURE_A_VERIFIER,146.206703,15.0,100.00000,3.0,OBSERVE_COUVERTURE_A_VERIFIER,1131.160547,0.0,0.00000,0.0,OBSERVE_COUVERTURE_A_VERIFIER,1398.237169,0.0,0.0,0.0,OBSERVE_COUVERTURE_A_VERIFIER
3,751072503,75056,Paris,Saint-Thomas d'Aquin 3,Habitat,1707.039002,20.186969,19.0,9.0,111.303842,44520.0,11.0,DISPONIBLE,2021.0,2022.0,239.496408,4.0,100.0,1.0,OBSERVE_COUVERTURE_A_VERIFIER,180.408966,2.0,99.70564,1.0,OBSERVE_COUVERTURE_A_VERIFIER,1094.156901,0.0,9.04358,0.0,OBSERVE_COUVERTURE_A_VERIFIER,1462.336075,0.0,0.0,0.0,OBSERVE_COUVERTURE_A_VERIFIER
4,751135029,75056,Paris,Gare 29,Habitat,3895.734065,20.899487,2.0,0.0,5.133821,23900.0,18.0,DISPONIBLE,2021.0,2022.0,278.496379,4.0,100.0,0.0,OBSERVE_COUVERTURE_A_VERIFIER,169.187855,4.0,100.00000,4.0,OBSERVE_COUVERTURE_A_VERIFIER,1151.780619,0.0,0.00000,0.0,OBSERVE_COUVERTURE_A_VERIFIER,1775.825865,0.0,0.0,0.0,OBSERVE_COUVERTURE_A_VERIFIER


## 6 — Préparer la table communale sans sommer les médianes


La population et la restauration sont agrégées seulement si tous les IRIS de la
commune disposent d’une valeur : une couverture incomplète reste indisponible.
Les lignes communales et les pourcentages de surface sont repris de J14.
Les médianes fiscales IRIS ne sont pas transformées en médianes communales.
Les mobilités de J13d restent à leur échelle communale et à leur millésime RP.


In [6]:
communes=pd.DataFrame({'CODGEO':communes_source['CODGEO'],'NOM_COMMUNE':communes_source['NOM_COMMUNE']})
groupes=iris.groupby('CODGEO')
def somme_complete(col):
    somme=groupes[col].sum(min_count=1)
    return somme.where(groupes[col].count().eq(groupes.size()))
for col in ['POPULATION','RESTAURANTS','RAPIDES']:communes[col]=communes['CODGEO'].map(somme_complete(col))
communes['DENSITE_RESTAURANTS']=communes['RESTAURANTS'].div(communes['POPULATION'].where(communes['POPULATION'].gt(0)))*10000
for col,source in {'MOBPRO_ENTRANTS':'J13D_MOBPRO_ENTRANTS','MOBPRO_SORTANTS':'J13D_MOBPRO_SORTANTS_FRANCE'}.items():communes[col]=extraire(communes_source,source)
for mode in meta14['rayons_metres']:
    for col,source in {f'LIGNES_{mode}':f'J14_NB_LIGNES_OBS_POINTS_REF_IRIS_{mode}',
        f'SURFACE_{mode}_PCT':f'J14_PART_SURFACE_COMMUNE_PROCHE_ARRET_OBS_{mode}_PCT',
        f'ARRETS_{mode}':f'J14_NB_ENTITES_ARRET_OBS_AFFECTEES_{mode}'}.items():communes[col]=extraire(communes_source,source)
if set(iris['CODGEO'])!=set(communes['CODGEO']):raise ValueError('Couverture IRIS/communes différente.')
exporter(communes,'profil_communes.csv')
display(communes.loc[communes['CODGEO'].isin(meta14['codes_focus'])])


,CODGEO,NOM_COMMUNE,POPULATION,RESTAURANTS,RAPIDES,DENSITE_RESTAURANTS,MOBPRO_ENTRANTS,MOBPRO_SORTANTS,LIGNES_METRO,SURFACE_METRO_PCT,ARRETS_METRO,LIGNES_BUS,SURFACE_BUS_PCT,ARRETS_BUS,LIGNES_RER,SURFACE_RER_PCT,ARRETS_RER,LIGNES_TRANSILIEN,SURFACE_TRANSILIEN_PCT,ARRETS_TRANSILIEN
1120,95127,Cergy,69578.0,208.0,138.0,29.894507,25678.690190,20588.965150,0.0,0.0,0.0,45.0,68.853573,161.0,1.0,34.513433,3.0,1.0,36.271165,3.0
1221,95500,Pontoise,31623.0,128.0,75.0,40.476868,11040.300707,10335.539743,0.0,0.0,0.0,25.0,72.564267,93.0,2.0,31.583644,1.0,3.0,38.475606,1.0


## 7 — Arrêts, lignes et relations de toute la région

Reprendre les exports régionaux J14. Les relations et les entités restent séparées.
La carte filtre à la demande les lignes qui ont un tracé ou des arrêts dans la
fenêtre de la commune/IRIS sélectionné. Les filtres ne recalculent pas les indicateurs.


In [7]:
arrets=lire_csv(FICHIER_ARRETS_REGION)
lignes=lire_csv(FICHIER_CATALOGUE_REGION)
rel=lire_csv(FICHIER_RELATIONS)
# Garder les lignes comportant un objet régional, avec leurs identifiants commerciaux.
utilisees=nombre(lignes['NB_ENTITES_ARRET']).gt(0)|nombre(lignes['NB_TRONCONS']).gt(0)
lignes=lignes.loc[utilisees].copy()
if not rel['ID_LINE'].isin(lignes['ID_LINE']).all():raise ValueError('Relations régionales absentes du catalogue.')
arrets=arrets.loc[arrets['ID_ARRET'].isin(rel['ID_ARRET']),['ID_ARRET','NOM_ARRET','LONGITUDE','LATITUDE']].copy()
arrets['NOM_ARRET']=arrets['NOM_ARRET'].fillna('Arrêt sans libellé diffusé')
if not arrets['ID_ARRET'].is_unique or not lignes['ID_LINE'].is_unique:raise ValueError('Objets régionaux dupliqués.')
if not rel['ID_ARRET'].isin(arrets['ID_ARRET']).all():raise ValueError('Relation avec un arrêt absent.')
if rel[['ID_ARRET','ID_LINE']].duplicated().any():raise ValueError('Relation arrêt–ligne dupliquée.')
exporter(arrets,'arrets.csv');exporter(lignes,'lignes.csv');exporter(rel,'relations.csv')
print('Arrêts régionaux :',len(arrets),' ; lignes avec objets :',len(lignes))


Arrêts régionaux : 34839  ; lignes avec objets : 1997


## 8 — Contours régionaux des communes et IRIS, tracés régionaux

Les codes des contours doivent correspondre exactement aux profils CSV. La commune
de contexte J14 est conservée, notamment pour Paris. Les contours sont simplifiés
uniquement pour l'affichage, en mètres dans EPSG:2154, puis exportés en EPSG:4326.
Les distances et les comptes restent ceux des traitements J14. Aucun IRIS n'est
supprimé pour réduire la taille. Les géométries détaillées restent dans processed.


In [8]:
def exporter_geo(obj,nom):
    p=DOSSIER_APP/nom
    p.write_text(json.dumps(obj,ensure_ascii=False,allow_nan=False,separators=(',',':')),encoding='utf-8')
    if p not in SORTIES:SORTIES.append(p)
    return p

def contours_affichage(fichier,couche,cle,codes):
    geo=gpd.read_file(fichier,layer=couche)
    if cle not in geo or geo.crs is None:raise ValueError(str(fichier)+' : clé ou CRS absent.')
    geo[cle]=geo[cle].astype('string').str.strip()
    if not geo[cle].is_unique or set(geo[cle])!=set(codes):raise ValueError(str(fichier)+' : contours différents des profils.')
    if geo.geometry.isna().any() or geo.geometry.is_empty.any() or not geo.geometry.is_valid.all():raise ValueError('Contours invalides : '+str(fichier))
    geo=geo[[cle,'geometry']].to_crs(2154)
    geo.geometry=geo.geometry.simplify(SIMPLIFICATION_CARTE_M,preserve_topology=True)
    if geo.geometry.is_empty.any() or not geo.geometry.is_valid.all():raise ValueError('Simplification invalide : '+str(fichier))
    return geo.to_crs(4326)

geo_communes=contours_affichage(FICHIER_GEO_COMMUNES,COUCHE_COMMUNES,'CODGEO',communes['CODGEO'])
geo_iris=contours_affichage(FICHIER_GEO_IRIS,COUCHE_IRIS,'IRIS',iris['IRIS'])
geo_iris=geo_iris.merge(iris[['IRIS','CODGEO']],on='IRIS',how='left',validate='one_to_one')
exporter_geo(json.loads(geo_communes.to_json(show_bbox=True)),'communes_idf.geojson')
exporter_geo(json.loads(geo_iris.to_json(show_bbox=True)),'iris_idf.geojson')
codes_geo=set(geo_iris['IRIS'])
traces_geo=gpd.read_file(FICHIER_TRACES_REGION,layer=COUCHE_TRACES)
if traces_geo.crs is None or 'ID_LINE' not in traces_geo:raise ValueError('Tracés régionaux : CRS ou ID_LINE absent.')
traces_geo['ID_LINE']=traces_geo['ID_LINE'].astype('string').str.strip()
if not traces_geo['ID_LINE'].isin(lignes['ID_LINE']).all():raise ValueError('Tracé régional sans ligne du catalogue.')
traces_geo=traces_geo[['ID_LINE','MODE','geometry']].to_crs(2154)
traces_geo.geometry=traces_geo.geometry.simplify(SIMPLIFICATION_CARTE_M,preserve_topology=True)
if traces_geo.geometry.isna().any() or traces_geo.geometry.is_empty.any():raise ValueError('Tracé régional vide.')
exporter_geo(json.loads(traces_geo.to_crs(4326).to_json(show_bbox=True)),'traces_idf.geojson')
# Conserver également les exports du focus pour la compatibilité des cartes J14.
for entree,sortie in [('iris_focus_j14.geojson','iris_focus.geojson'),('traces_focus_j14.geojson','traces_focus.geojson')]:
    exporter_geo(json.loads((A14/entree).read_text(encoding='utf-8')),sortie)
print('Contours régionaux :',len(geo_communes),'communes ;',len(geo_iris),'IRIS.')
print('Tronçons régionaux :',len(traces_geo))


Contours régionaux : 1267 communes ; 5264 IRIS.
Tronçons régionaux : 9554


## 9 — Restaurants géolocalisés dans tous les IRIS du profil

Les restaurants avec coordonnées valides et affectation à un IRIS du profil sont
exportés pour toute la région. La commune est reprise de la commune de contexte
J14 ; elle n'est pas reconstruite arbitrairement depuis les cinq premiers chiffres.
Les observations non affectées ne sont pas transformées en restaurants absents.
Les noms disponibles restent au survol. Le plafond d'affichage de 5 000 marqueurs
par carte est annoncé dans l'application et n'affecte aucun compte statistique.


In [9]:
restaurants=pd.DataFrame(columns=['CLE_POINT','NOM','CATEGORIE','LONGITUDE','LATITUDE','IRIS_AFFECTE','CODGEO'])
if CHARGER_RESTAURANTS:
    points=lire_csv(FICHIER_POINTS)
    requis={'SOURCE','CLE_POINT','NOM','CATEGORIE','IRIS_AFFECTE','COORDONNEES_VALIDES','LONGITUDE','LATITUDE'}
    if requis-set(points):raise ValueError('Points J13c : colonnes absentes '+str(sorted(requis-set(points))))
    ok=points['SOURCE'].eq('RESTAURATION')&points['IRIS_AFFECTE'].isin(codes_geo)&points['COORDONNEES_VALIDES'].str.lower().isin(['true','1'])
    restaurants=points.loc[ok,['CLE_POINT','NOM','CATEGORIE','LONGITUDE','LATITUDE','IRIS_AFFECTE']].copy()
    restaurants['CODGEO']=restaurants['IRIS_AFFECTE'].map(iris.set_index('IRIS')['CODGEO'])
    restaurants['NOM']=restaurants['NOM'].fillna('Libellé non diffusé');restaurants['CATEGORIE']=restaurants['CATEGORIE'].fillna('Type non renseigné')
    if not restaurants['CLE_POINT'].is_unique:raise ValueError('Restaurants géolocalisés : clé dupliquée.')
exporter(restaurants,'restaurants.csv')
print('Restaurants géolocalisés régionaux',len(restaurants))


Restaurants géolocalisés régionaux 47856


## 10 — Sources et dates affichées dans l’application


Le registre existant est utilisé s’il est présent. Les sources IDFM et le contexte
INSEE amont sont ajoutés en complément sans inventer de date. Les dates de la
population, des revenus et du réseau peuvent être différentes : elles doivent
rester visibles. Les chemins Windows personnels ne sont pas inclus dans l’interface.

Le registre accepte le séparateur virgule ou point-virgule. Un registre existant
mais illisible provoque une erreur explicite ; ses lignes ne sont pas remplacées
silencieusement par une table vide.


In [10]:
cols_sources=['fichier','source','annee_donnees','annee_geographie','date_telechargement','url']
sources=lire_registre(FICHIER_REGISTRE,SEPARATEUR_REGISTRE)[cols_sources] if FICHIER_REGISTRE.is_file() else pd.DataFrame(columns=cols_sources)
sources['fichier']=sources['fichier'].astype('string').str.replace('\\','/',regex=False).str.rsplit('/',n=1).str[-1]
deja=set(zip(sources['fichier'].fillna(''),sources['url'].fillna('')))
libelles_sources={'ARRETS':'IDFM — Arrêts et lignes associées','TRACES_FERRES':'IDFM — Tracés géographiques du réseau ferré',
    'TRACES_BUS':'IDFM — Tracés des lignes régulières de bus','REFERENTIEL':'IDFM — Référentiel des lignes commerciales'}
rows=[]
for cfg in meta14['sources']:
    if not cfg['actif']:continue
    if (Path(cfg['fichier']).name,cfg['url']) in deja:continue
    rows.append({'fichier':Path(cfg['fichier']).name,'source':libelles_sources[cfg['nom']],
        'annee_donnees':cfg['annee_donnees'],'annee_geographie':cfg['annee_geographie'],
        'date_telechargement':cfg['date_telechargement'],'url':cfg['url']})
rows.append({'fichier':FICHIER_IRIS.name,'source':'INSEE et sources amont — profils IRIS repris de '+meta14['etape_base'],
    'annee_donnees':meta14['annee_rp'],'annee_geographie':meta14['geographie_iris'],'date_telechargement':None,'url':None})
prov_c=json.loads(FICHIER_META_C.read_text(encoding='utf-8')).get('provenance_sources_declarees',{}) if FICHIER_META_C.is_file() else {}
bpe=prov_c.get('RESTAURATION',{})
rows.append({'fichier':FICHIER_POINTS.name,'source':'INSEE — BPE, observations de restauration préparées dans J13c',
    'annee_donnees':bpe.get('annee_source'),'annee_geographie':bpe.get('geographie_source'),
    'date_telechargement':None,'url':bpe.get('url_notice')})
def millesime_unique(col):
    v=iris[col].dropna().unique()
    return str(int(v[0])) if len(v)==1 and v[0]==int(v[0]) else None
rows.append({'fichier':FICHIER_IRIS.name,'source':'INSEE — Filosofi, indicateurs fiscaux IRIS hérités',
    'annee_donnees':millesime_unique('ANNEE_REVENUS'),'annee_geographie':millesime_unique('GEOGRAPHIE_REVENUS'),
    'date_telechargement':None,'url':None})
sources=pd.concat([sources.astype('string'),pd.DataFrame(rows,columns=cols_sources).astype('string')],ignore_index=True).drop_duplicates().reset_index(drop=True)
# Éviter de publier un chemin personnel si un ancien registre en contient.
sources['fichier']=sources['fichier'].astype('string').str.replace('\\','/',regex=False).str.rsplit('/',n=1).str[-1]
exporter(sources,'sources.csv')
exporter(sources,'sources_application_j15a.csv',sep=';',dossier=DOSSIER_DOCS)
display(sources)


Séparateur du registre : ','


,fichier,source,annee_donnees,annee_geographie,date_telechargement,url
0,population_communes_2022.csv,INSEE,2022,2025,2026-08-12,https://www.insee.fr/fr/statistiques/8581696
1,arrets_lignes_idfm.csv,Île-de-France Mobilités - Arrêts et lignes ass...,2026,2026,2026-08-24,https://data.iledefrance-mobilites.fr/explore/...
2,traces_reseau_ferre_idfm.geojson,Île-de-France Mobilités - Tracés du réseau ferré,2024,2024,2026-08-24,https://data.iledefrance-mobilites.fr/explore/...
3,DS_BPE_CSV_FR.zip,INSEE - Base permanente des équipements 2025,2025,2026,2026-08-24,https://www.insee.fr/fr/statistiques/8217527?s...
4,bpe_2025_detail.csv,Insee - Base permanente des équipements,2025,2025,2026-08-25,https://www.insee.fr/fr/statistiques/8217525
5,population_iris_2022.xlsx,Insee - Population à l'IRIS,2022,2024,2026-10-07,https://www.insee.fr/fr/statistiques/8647014
6,menages_familles_iris_2022.xlsx,Insee - Couples familles et ménages à l'IRIS,2022,2024,2026-10-07,https://www.insee.fr/fr/statistiques/8647008
7,activite_iris_2022.xlsx,Insee - Activité des résidents à l'IRIS,2022,2024,2026-10-07,https://www.insee.fr/fr/statistiques/8647006
8,diplomes_iris_2022.xlsx,Insee - Diplômes et formation à l'IRIS,2022,2024,2026-10-07,https://www.insee.fr/fr/statistiques/8647010
9,logement_iris_2022.xlsx,Insee - Logement à l'IRIS,2022,2024,2026-10-07,https://www.insee.fr/fr/statistiques/8647012


## 11 — Manifeste des données et contrôles de tailles


Le manifeste fournit les dates, le focus, les rayons et les empreintes des petits
exports. L’application vérifie ces empreintes au chargement. Le seuil 20 Mo est
un choix de préparation par fichier, pas une limite annoncée du service Streamlit.
Les données doivent être régénérées ensemble lors d’une mise à jour.


In [11]:
fichiers_app=[p for p in SORTIES if p.parent==DOSSIER_APP]
tailles=pd.DataFrame([{'nom':p.name,'octets':p.stat().st_size,'sha256':sha(p)} for p in fichiers_app])
if tailles['nom'].duplicated().any():raise ValueError('Sorties dupliquées : relancer depuis le bloc 3.')
if tailles['octets'].gt(MAX_FICHIER_MO*1000000).any():raise ValueError('Un export dépasse le seuil choisi. Réduire les champs/géométries d’affichage, pas les IRIS des tables.')
manifest={'schema_version':2,'etape':'J15A_APPLICATION','statut':'PROTOTYPE_DESCRIPTIF',
    'generation_utc':datetime.now(timezone.utc).isoformat(),'annee_rp':meta14['annee_rp'],'geographie_iris':meta14['geographie_iris'],
    'etape_base':meta14['etape_base'],'nb_iris':NOMBRE_IRIS,'nb_communes':len(communes),'cartographie_regionale':True,'simplification_carte_metres':SIMPLIFICATION_CARTE_M,'codes_focus':meta14['codes_focus'],
    'noms_focus':[nom_commune.loc[c] for c in meta14['codes_focus']],
    'rayons_metres':meta14['rayons_metres'],'couverture_arrets_confirmee_idf':meta14['couverture_arrets_confirmee_idf'],
    'plafond_bus_carte_j14':meta14['plafond_bus_carte'],'restaurants_geolocalises_charges':CHARGER_RESTAURANTS,
    'empreinte_metadata_j14':sha(FICHIER_META),'limites':meta14['limites'],'fichiers':tailles.to_dict('records')}
FICHIER_MANIFEST=DOSSIER_APP/'manifest.json'
FICHIER_MANIFEST.write_text(json.dumps(manifest,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
exporter(tailles,'tailles_exports_j15a.csv',dossier=DOSSIER_INTERIM)
exporter(pd.DataFrame(DIAGNOSTIC,columns=['COLONNE_SOURCE','STATUT']),'indicateurs_absents_j15a.csv',dossier=DOSSIER_INTERIM)
print('Exports app',round(tailles['octets'].sum()/1e6,2),'Mo ; IRIS',NOMBRE_IRIS)


Exports app 30.73 Mo ; IRIS 5264


## 12 — Valider les exports avec le lecteur de l’application


Ce contrôle utilise le même lecteur que Streamlit. Il vérifie les clés, les
relations, les nombres, les pourcentages et les empreintes. Il ne remplace pas
la vérification des sources ni une comparaison avec un arrêt connu sur le terrain.


In [12]:
if str(RACINE) not in sys.path:sys.path.insert(0,str(RACINE))
from app.core import load_bundle,client_table,fiche
jeu=load_bundle(DOSSIER_APP)
assert set(jeu['profil_iris']['IRIS'])==ATTENDUS
assert len(jeu['profil_iris'])==NOMBRE_IRIS
assert set(jeu['profil_communes']['CODGEO'])==set(communes_source['CODGEO'])
assert 'MEDIANE_UC' not in jeu['profil_communes']
assert set(jeu['_communes_geo'])==set(communes['CODGEO'])
assert set(jeu['_iris_geo'])==ATTENDUS
display(client_table(jeu['profil_iris'].head(),['NOM_IRIS','POPULATION','RESTAURANTS','MEDIANE_UC','PAUVRETE_PCT']))
print('Exports lus et validés par le code de l’application.')


,Quartier / IRIS,Population,Restaurants observés,Niveau de vie médian annuel par UC (€),Taux de pauvreté (%)
0,Les Archives 1,2317.001126,16.0,37610.0,12.0
1,Saint-Victor 2,1665.564957,22.0,37090.0,11.0
2,Notre-Dame des Champs 9,2414.995652,25.0,35410.0,13.0
3,Saint-Thomas d'Aquin 3,1707.039002,19.0,44520.0,11.0
4,Gare 29,3895.734065,2.0,23900.0,18.0


Exports lus et validés par le code de l’application.


## 13 — Documenter J15a et ses limites


La documentation permet de comprendre l’application sans ouvrir les notebooks
précédents. Elle précise les sources et les différences de millésimes. J15b est
annoncé comme un enrichissement à venir, pas comme un calcul déjà effectué.


In [13]:
doc="""# J15a — Application descriptive de géomarketing

## Objectifs
Explorer communes et IRIS, comparer population/restauration, présenter les revenus
disponibles, les mobilités communales et le réseau métro/bus/RER/Transilien.

## Données et périmètre
Exports J14 contrôlés ; tous les IRIS et communes du profil IDF dans les tables et les cartes.
CSV et GeoJSON dans data/app/j15 ; dates, rayons et empreintes dans manifest.json.
Les profils complets restent dans processed. Les médianes fiscales restent IRIS.

## Application
Entrée : app/streamlit_app.py ; dépendances : app/requirements.txt.
Lancer depuis la racine : python -m streamlit run app/streamlit_app.py.
Navigation : clic commune → sous-couche IRIS → clic IRIS → fiche détaillée.
Les menus latéraux et les cartes partagent la même sélection. Une sélection
hors du département courant met à jour le filtre. Vue IDF permet de revenir
à la carte régionale. Les transports et restaurants reprennent les exports régionaux.
Vues : communes, IRIS, transports, sources et méthode. Fiches et CSV téléchargeables.
Couleurs/pictogrammes disponibles intégrés ; aucun fond externe ni clé API.
Contours simplifiés uniquement pour l'affichage, avec conservation de tous les codes.
Registre des sources : séparateur virgule ou point-virgule détecté automatiquement.

## Interprétation
Diagnostic descriptif. Les rayons sont directs ; pas des temps de marche.
Surface proche des arrêts, pas population desservie. Le tracé ne prouve pas une desserte.
Les flux de travail restent communaux et ne sont pas affectés aux lignes.
Les valeurs absentes ne sont pas des zéros. Aucun chiffre d’affaires n’est prédit.

## Validation avant démonstration
Contrôles J14 et J15a, affichage de tous les territoires, filtres, données absentes,
cartes et téléchargements à vérifier sur les fichiers réels du projet.

## Suite
J15b : données GTFS pour fréquence/amplitude sur une semaine définie.
Puis validation d’une première mise en ligne et retours sur la lisibilité.
Une étude d’emplacement précise demande accès piétons, données de fréquentation
si disponibles, observation terrain et caractéristiques commerciales du site.
"""
(DOSSIER_DOCS/'j15a.md').write_text(doc,encoding='utf-8')
print('Documentation créée',DOSSIER_DOCS/'j15a.md')


Documentation créée C:\Users\almou\OneDrive\GeoMarketing_IDF\docs\j15a.md


## 14 — Installer les dépendances de l’application


Dans **Anaconda Prompt**, aller à la racine du projet. Une installation séparée
évite de modifier l’environnement des notebooks. Les versions fournies ont été
utilisées pour les contrôles de ce livrable ; choisir la même version Python lors
du déploiement. Le notebook ne lance pas cette installation automatiquement.

```shell
cd /d C:\Users\almou\OneDrive\GeoMarketing_IDF
conda create -n geomarketing_app python=3.12 pip
conda activate geomarketing_app
python -m pip install -r app/requirements.txt
```

Si l’environnement existe déjà : seulement conda activate, puis pip install.
Ne pas changer une installation fonctionnelle sans besoin. Le thème facultatif
est dans app/theme.example.toml : copier ses réglages dans .streamlit/config.toml
en conservant les réglages déjà présents.


## 15 — Lancer Streamlit en local


Toujours depuis la racine du projet, dans Anaconda Prompt :

```shell
conda activate geomarketing_app
python -m streamlit run app/streamlit_app.py
```

Ouvrir l’URL affichée, généralement http://localhost:8501. Garder le terminal
ouvert tant que l’application tourne. Ctrl+C arrête le serveur local.
Un notebook prépare les données ; Streamlit tourne dans ce processus distinct.
Il n’est pas nécessaire de recopier les grands CSV bruts dans l’application.


## 16 — Vérifier le parcours utilisateur


Effectuer cette lecture avant une démonstration sur vos données réelles :

1. Sur la carte IDF, cliquer sur une commune hors Cergy–Pontoise : vérifier le zoom et ses IRIS.
2. Cliquer sur un IRIS : vérifier la commune, le code IRIS, le changement de vue et la fiche.
3. Revenir à la commune puis à la vue IDF ; essayer un autre département.
4. Choisir une commune/IRIS dans les menus : vérifier la synchronisation avec la carte.
5. Comparer les populations et comptes à J14, notamment Cergy et Pontoise.
2. Choisir un autre département et vérifier les vues Communes et IRIS.
3. Lire un IRIS avec revenus disponibles, puis un IRIS sans valeur publiée.
4. Vérifier que Non disponible apparaît plutôt qu’un zéro artificiel.
5. Dans Transports, filtrer le bus, le RER, le Transilien et le métro ; changer les lignes.
6. Cliquer les pictogrammes pour masquer/afficher les tracés ; lire les noms au survol.
7. Désactiver les restaurants ; les comptes statistiques doivent rester identiques.
8. Télécharger une fiche et un CSV ; vérifier unités, dates et accents.
9. Ouvrir Sources et méthode ; compléter les dates connues dans le registre, puis régénérer.

Le ZIP inclut le composant cartographique prêt à l'emploi : aucune compilation
JavaScript n'est nécessaire chez l'utilisateur. Les clics restent utilisables
sans API key et les logos sont conservés sous la carte.

Les contrôles automatisés du livrable utilisent des fichiers de test distincts.
Ils ne certifient pas l’exactitude de vos téléchargements ou des observations terrain.


## 17 — Quels fichiers mettre sur GitHub ?


Versionner app/, le notebook et docs/j15a.md, puis **uniquement les exports
préparés data/app/j15** nécessaires à cette démonstration. Les fichiers bruts,
interim et processed restent hors dépôt comme dans J14.

Les données locales Windows ne seront pas accessibles au serveur Streamlit :
les exports doivent être présents dans le dépôt déployé, ou dans un stockage
distant explicitement configuré. Pour cette version, les petits exports publics
dans le dépôt constituent le parcours le plus simple.

Si data/app est ignoré, git add -f cible ici uniquement les exports générés J15a :

```shell
git status --short
git add app/ notebooks/15a_preparer_application_streamlit_idf.ipynb docs/j15a.md docs/sources_application_j15a.csv
git add -f data/app/j15/
git diff --cached --stat
git commit -m "feat: ajouter une application Streamlit de diagnostic territorial"
git push
```

Si app/ existait déjà, intégrer les nouveaux fichiers en conservant ses autres
fonctions. Ne pas remplacer les réglages existants ni un requirements sans lecture.
Supprimer les outputs du notebook avant de l’enregistrer dans le dépôt.


## 18 — Première mise en ligne sur Streamlit Community Cloud


Après vérification locale, ouvrir https://share.streamlit.io/. Depuis **Create app**,
renseigner le dépôt GitHub, la branche et le chemin **app/streamlit_app.py**.
Le fichier **app/requirements.txt**, à côté de l’entrée, contient les dépendances.
Choisir Python 3.12 pour rester cohérent avec l’environnement prévu, puis lancer Deploy.

Community Cloud exécute l’application depuis la racine du dépôt. Les chemins
du code sont donc relatifs au projet et utilisent pathlib, sans chemin Windows.
Il n’est pas nécessaire de créer un serveur personnel pour cette démonstration.

Après déploiement : vérifier toutes les vues dans l’URL distante, puis comparer
une commune connue aux valeurs locales. Une modification GitHub de l’application
ou des petits exports constitue ensuite une mise à jour à vérifier.

Documentation officielle :
- https://docs.streamlit.io/deploy/streamlit-community-cloud/deploy-your-app/deploy
- https://docs.streamlit.io/deploy/streamlit-community-cloud/deploy-your-app/app-dependencies
- https://docs.streamlit.io/deploy/streamlit-community-cloud/deploy-your-app/file-organization

Ce notebook ne publie rien ni ne demande d’accès à votre compte GitHub.


## 19 — Quand commencer l’application et quand la montrer ?


| Étape | Résultat attendu |
|---|---|
| J15a, dès maintenant si J14 est validé | Application locale avec données réelles et cartes |
| Fin J15a, après lecture des filtres/cartes/sources | Première mise en ligne possible pour une démonstration descriptive |
| J15b | Horaires et fréquences GTFS ajoutés dans la même application |
| Après retours utilisateurs et étude terrain ciblée | Démonstration plus solide pour proposer une étude d’implantation |

Les J sont des étapes de travail, pas une promesse de durée calendaire.
Le calendrier dépend des fichiers présents, de leurs contrôles et du temps disponible.
Pour un client, présenter ce que l’application mesure effectivement et des exemples
de questions auxquelles elle répond. Le déploiement seul ne constitue pas une mission vendue.


## 20 — J15b : préparer les horaires et fréquences GTFS


La suite transports proposée après J14 reste prévue. Les données d’horaires sont
distinctes des arrêts et tracés statiques déjà utilisés. Elles demandent un fichier
GTFS et un choix explicite de dates : ne pas reconstituer une fréquence à partir du
nombre d’arrêts d’une ligne.

Prévoir un notebook séparé : **15b_analyser_frequences_gtfs_idf.ipynb**.

1. Enregistrer le GTFS original dans raw et noter la date et l’URL réelles.
2. Inspecter stops, routes, trips, stop_times, calendar et calendar_dates ; vérifier frequencies si présent.
3. Choisir une semaine et des plages horaires ; appliquer les exceptions de calendrier.
4. Conserver les identifiants de service, course, ligne et arrêt ; vérifier leur correspondance IDFM.
5. Compter les passages programmés aux arrêts, par date, sens et plage horaire.
6. Distinguer les passages programmés des circulations réellement observées et des voyageurs.
7. Documenter les cas particuliers GTFS, notamment horaires au-delà de 24 h et services définis par fréquences.
8. Ajouter les résultats disponibles et leur date à l’application, sans remplacer les anciens indicateurs spatiaux.

J15b n’est pas calculé dans ce livrable : aucun GTFS réel n’a été fourni.
La marche réelle et la fréquentation restent ensuite des analyses distinctes.
